# BIST basket monthly rebalance — explained strategies v14

Bu sürüm önceki chart-first not defterini düzeltir.

En kritik düzeltme:
- **Equal**
- **HRP**
- **BL**
- **HYBRID**

stratejileri geri geldi. Yani artık sadece tek bir optimizer akışı yok; stratejileri yan yana karşılaştırabiliyorsun.

Amaç:
- geniş BIST sepeti + döviz + kıymetli madenler ile deney yapmak
- her ay yeni portföy kurmak
- stratejileri aynı veri üzerinde karşılaştırmak
- her grafik için neyi okuman gerektiğini açıkça yazmak

In [1]:
!pip -q install yfinance PyPortfolioOpt plotly pandas numpy scipy scikit-learn openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 7.6 MB/s eta 0:00:00


In [2]:
import warnings, json, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go

from pypfopt.risk_models import CovarianceShrinkage
from pypfopt.hierarchical_portfolio import HRPOpt
from pypfopt.black_litterman import BlackLittermanModel
from pypfopt.efficient_frontier import EfficientFrontier

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

## 1) Universe config

Bu hücrede yatırım evreni tanımlanır.

İçerik:
- BIST hisseleri
- USDTRY, EURTRY
- ALTIN_TRY, GUMUS_TRY, PLATIN_TRY

Dürüst not:
Aşağıdaki BIST listesi önceki geniş evren denemesinden alınmış başlangıç listesidir.
Bunu canlı doğrulanmış güncel resmi BIST100 listesi gibi sunmuyorum.

In [3]:
BIST_TICKERS = ['AEFES.IS', 'AGHOL.IS', 'AKBNK.IS', 'AKFYE.IS', 'AKSA.IS', 'AKSEN.IS', 'ALARK.IS', 'ARCLK.IS', 'ASELS.IS', 'ASTOR.IS', 'BIMAS.IS', 'BOBET.IS', 'CCOLA.IS', 'CIMSA.IS', 'CWENE.IS', 'DOAS.IS', 'DOHOL.IS', 'ECILC.IS', 'EGEEN.IS', 'EKGYO.IS', 'ENERY.IS', 'ENJSA.IS', 'ENKAI.IS', 'EREGL.IS', 'EUPWR.IS', 'FROTO.IS', 'GARAN.IS', 'GESAN.IS', 'GUBRF.IS', 'HALKB.IS', 'HEKTS.IS', 'ISCTR.IS', 'ISMEN.IS', 'KAYSE.IS', 'KCAER.IS', 'KCHOL.IS', 'KLSER.IS', 'KOZAA.IS', 'KOZAL.IS', 'KRDMD.IS', 'MAVI.IS', 'MGROS.IS', 'MIATK.IS', 'ODAS.IS', 'OTKAR.IS', 'OYAKC.IS', 'PETKM.IS', 'PGSUS.IS', 'REEDR.IS', 'SASA.IS', 'SAHOL.IS', 'SDTTR.IS', 'SISE.IS', 'SKBNK.IS', 'SMRTG.IS', 'SOKM.IS', 'TAVHL.IS', 'TCELL.IS', 'THYAO.IS', 'TKFEN.IS', 'TOASO.IS', 'TSKB.IS', 'TTKOM.IS', 'TTRAK.IS', 'TUPRS.IS', 'ULKER.IS', 'VAKBN.IS', 'VESBE.IS', 'VESTL.IS', 'YKBNK.IS', 'ZOREN.IS']

FX_TICKERS = {'USDTRY': 'USDTRY=X', 'EURTRY': 'EURTRY=X'}
METAL_USD_TICKERS = {'ALTIN_USD': 'GC=F', 'GUMUS_USD': 'SI=F', 'PLATIN_USD': 'PL=F'}

CONFIG = {
    'salary_try': 25000,
    'target_test_months': 6,
    'target_lookback_days': 252,
    'download_period': '10y',
    'min_history_days': 180,
    'max_weight': 0.10,
    'gold_label': 'ALTIN_TRY',
    'gold_min': 0.05,
    'gold_max': 0.20,
    'hybrid_alpha': 0.25,
    'strategies': ['Equal', 'HRP', 'BL', 'HYBRID'],
}

print('BIST başlangıç evreni:', len(BIST_TICKERS))
print('Stratejiler:', CONFIG['strategies'])

BIST başlangıç evreni: 71
Stratejiler: ['Equal', 'HRP', 'BL', 'HYBRID']


## 2) Helper functions

Bu bölüm veri indirme, temizlik, format ve strateji yardımcılarını içerir.

Okurken önemli olan:
- burada sadece alt yapı var
- asıl yorum yapılacak bölümler daha aşağıda

In [4]:
def try_fmt(x):
    if pd.isna(x):
        return '-'
    return f'₺{x:,.0f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

def pct_fmt(x):
    if pd.isna(x):
        return '-'
    return f'{x*100:.2f}%'

def safe_asset_name(symbol):
    return symbol.replace('.IS', '').replace('=X', '').replace('=F', '')

def fetch_close_series(symbol, period):
    df = yf.download(symbol, period=period, auto_adjust=True, progress=False)
    if df is None or len(df) == 0:
        return None
    if isinstance(df.columns, pd.MultiIndex):
        level0 = list(df.columns.get_level_values(0))
        close = df.xs('Close', axis=1, level=0) if 'Close' in level0 else df.iloc[:, :1]
    else:
        close = df[['Close']] if 'Close' in df.columns else df.iloc[:, :1]
    if isinstance(close, pd.DataFrame):
        if close.shape[1] == 0:
            return None
        close = close.iloc[:, 0]
    s = pd.Series(close).dropna()
    s.name = symbol
    return s if len(s) else None

def normalize_prices(df):
    return df / df.iloc[0] * 100

def clean_returns(df):
    rets = df.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='any')
    if rets.empty:
        return rets
    valid_cols = [c for c in rets.columns if np.isfinite(rets[c]).all() and rets[c].std() > 1e-10]
    if len(valid_cols) == 0:
        return pd.DataFrame(index=rets.index)
    return rets[valid_cols]

def normalize_weights(w, index):
    w = pd.Series(w).reindex(index).fillna(0.0).astype(float)
    s = w.sum()
    if s <= 0 or not np.isfinite(s):
        return pd.Series(1/len(index), index=index)
    return w / s

def band_gold(weights, gold_label, gold_min, gold_max):
    w = weights.copy().astype(float)
    if gold_label not in w.index:
        return normalize_weights(w, w.index)
    g = float(w[gold_label])
    tg = min(max(g, gold_min), gold_max)
    if abs(tg - g) < 1e-12:
        return normalize_weights(w, w.index)
    rest = w.drop(gold_label)
    if rest.sum() <= 0:
        w[:] = 0.0
        w[gold_label] = 1.0
        return w
    rest = rest / rest.sum()
    w[gold_label] = tg
    w.loc[rest.index] = (1 - tg) * rest
    return normalize_weights(w, w.index)

def equal_weight(columns):
    return pd.Series(1 / len(columns), index=columns)

def inverse_vol_weight(price_window):
    rets = clean_returns(price_window)
    if rets.empty or rets.shape[1] == 0:
        return equal_weight(price_window.columns)
    iv = 1 / rets.std().replace(0, np.nan)
    iv = iv.replace([np.inf, -np.inf], np.nan).dropna()
    if iv.empty:
        return equal_weight(price_window.columns)
    w = pd.Series(0.0, index=price_window.columns)
    w.loc[iv.index] = iv / iv.sum()
    return normalize_weights(w, price_window.columns)

def hrp_weights_safe(price_window):
    rets = clean_returns(price_window)
    if rets.shape[0] < 2 or rets.shape[1] < 2:
        return inverse_vol_weight(price_window)
    try:
        hrp = HRPOpt(rets)
        w_sub = pd.Series(hrp.optimize()).reindex(rets.columns).fillna(0.0)
        w = pd.Series(0.0, index=price_window.columns)
        w.loc[w_sub.index] = w_sub
        return normalize_weights(w, price_window.columns)
    except Exception as e:
        print('HRP failed, inverse-vol kullanıldı:', str(e))
        return inverse_vol_weight(price_window)

def views_from_prices(price_window):
    rets = clean_returns(price_window)
    if rets.empty or rets.shape[1] == 0:
        return {c: 0.0 for c in price_window.columns}
    ann = rets.mean() * 252
    vol = rets.std() * np.sqrt(252)
    score = (ann / vol.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0)
    score = score.reindex(price_window.columns).fillna(0.0)
    return score.clip(-0.15, 0.15).to_dict()

def bl_weights_safe(price_window):
    rets = clean_returns(price_window)
    if rets.shape[0] < 2 or rets.shape[1] < 2:
        w = inverse_vol_weight(price_window)
        diag = pd.DataFrame({'asset': list(price_window.columns), 'view': [0.0]*len(price_window.columns), 'view_source': 'rules_insufficient_data'})
        return w, diag
    try:
        S = CovarianceShrinkage(rets, returns_data=True).ledoit_wolf()
        views = views_from_prices(price_window)
        bl = BlackLittermanModel(S, pi='equal', absolute_views=views, omega='default')
        post = bl.bl_returns()
        ef = EfficientFrontier(post, S, weight_bounds=(0, CONFIG['max_weight']))
        ef.max_sharpe(risk_free_rate=0.0)
        w = pd.Series(ef.clean_weights()).reindex(price_window.columns).fillna(0.0)
        w = normalize_weights(w, price_window.columns)
        w = band_gold(w, CONFIG['gold_label'], CONFIG['gold_min'], CONFIG['gold_max'])
        diag = pd.DataFrame({'asset': list(views.keys()), 'view': list(views.values()), 'view_source': 'rules'})
        return w, diag
    except Exception as e:
        print('BL failed, inverse-vol kullanıldı:', str(e))
        w = inverse_vol_weight(price_window)
        fallback_views = views_from_prices(price_window)
        diag = pd.DataFrame({'asset': list(fallback_views.keys()), 'view': list(fallback_views.values()), 'view_source': 'rules_bl_fallback'})
        return w, diag

def hybrid_weights(price_window):
    w_hrp = hrp_weights_safe(price_window)
    w_bl, diag = bl_weights_safe(price_window)
    a = CONFIG['hybrid_alpha']
    w = (1 - a) * w_hrp + a * w_bl
    w = normalize_weights(w, price_window.columns)
    w = w.clip(lower=0, upper=CONFIG['max_weight'])
    w = normalize_weights(w, price_window.columns)
    w = band_gold(w, CONFIG['gold_label'], CONFIG['gold_min'], CONFIG['gold_max'])
    return w, diag

def compute_weights(strategy_name, price_window):
    if strategy_name == 'Equal':
        return equal_weight(price_window.columns), None
    if strategy_name == 'HRP':
        return hrp_weights_safe(price_window), None
    if strategy_name == 'BL':
        return bl_weights_safe(price_window)
    if strategy_name == 'HYBRID':
        return hybrid_weights(price_window)
    raise ValueError(f'Unknown strategy: {strategy_name}')

## 3) Data coverage and eligibility

Bu bölüm en kritik güven kontrolüdür.

Bakman gereken şeyler:
- veri gerçekten inmiş mi
- hangi varlıklar testte kalmış
- ortak tarihçe yeterli mi

Eğer burada kalite zayıfsa, aşağıdaki bütün performans sonuçları anlamsız hale gelir.

In [5]:
rows = []
series_list = []

for sym in BIST_TICKERS:
    s = fetch_close_series(sym, CONFIG['download_period'])
    rows.append({'source': sym, 'asset': safe_asset_name(sym), 'group': 'BIST', 'n': 0 if s is None else len(s), 'start': None if s is None else str(s.index.min().date()), 'end': None if s is None else str(s.index.max().date()), 'status': 'ok' if s is not None else 'empty'})
    if s is not None and len(s) >= CONFIG['min_history_days']:
        s = s.copy(); s.name = safe_asset_name(sym); series_list.append(s)

fx_series = {}
for label, sym in FX_TICKERS.items():
    s = fetch_close_series(sym, CONFIG['download_period'])
    fx_series[label] = s
    rows.append({'source': sym, 'asset': label, 'group': 'FX', 'n': 0 if s is None else len(s), 'start': None if s is None else str(s.index.min().date()), 'end': None if s is None else str(s.index.max().date()), 'status': 'ok' if s is not None else 'empty'})
    if s is not None and len(s) >= CONFIG['min_history_days']:
        s = s.copy(); s.name = label; series_list.append(s)

metal_usd = {}
for label, sym in METAL_USD_TICKERS.items():
    s = fetch_close_series(sym, CONFIG['download_period'])
    metal_usd[label] = s
    rows.append({'source': sym, 'asset': label, 'group': 'METAL_USD', 'n': 0 if s is None else len(s), 'start': None if s is None else str(s.index.min().date()), 'end': None if s is None else str(s.index.max().date()), 'status': 'ok' if s is not None else 'empty'})

usdtry = fx_series.get('USDTRY')
for try_label, usd_label in [('ALTIN_TRY', 'ALTIN_USD'), ('GUMUS_TRY', 'GUMUS_USD'), ('PLATIN_TRY', 'PLATIN_USD')]:
    base = metal_usd.get(usd_label)
    if base is not None and usdtry is not None:
        s = (base * usdtry).dropna()
        s.name = try_label
        rows.append({'source': f'{usd_label} * USDTRY', 'asset': try_label, 'group': 'METAL_TRY', 'n': len(s), 'start': str(s.index.min().date()), 'end': str(s.index.max().date()), 'status': 'ok'})
        if len(s) >= CONFIG['min_history_days']:
            series_list.append(s)

coverage = pd.DataFrame(rows)
display(coverage.head())

prices = pd.concat(series_list, axis=1).sort_index().ffill()
common_start = prices.apply(lambda s: s.first_valid_index()).max()
prices = prices.loc[common_start:].dropna()
returns = prices.pct_change().dropna()

eligible_assets = pd.DataFrame({'asset': prices.columns, 'group': [coverage.set_index('asset').loc[a, 'group'] if a in coverage['asset'].values else 'OTHER' for a in prices.columns]})

monthly_points = prices.resample('MS').first().dropna()
lookback = min(CONFIG['target_lookback_days'], max(60, len(prices) - 40))
months = CONFIG['target_test_months']

print('Ortak başlangıç:', common_start)
print('Kullanılan varlık sayısı:', prices.shape[1])
print('Toplam gün:', len(prices), 'lookback:', lookback, 'test_months:', months)
display(prices.tail())

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: KOZAA.IS"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['KOZAA.IS']: YFPricesMissingError('possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: KOZAL.IS"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['KOZAL.IS']: YFPricesMissingError('possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")')


,source,asset,group,n,start,end,status
0,AEFES.IS,AEFES,BIST,2543,2016-03-30,2026-03-30,ok
1,AGHOL.IS,AGHOL,BIST,2543,2016-03-30,2026-03-30,ok
2,AKBNK.IS,AKBNK,BIST,2542,2016-03-30,2026-03-30,ok
3,AKFYE.IS,AKFYE,BIST,761,2023-03-16,2026-03-30,ok
4,AKSA.IS,AKSA,BIST,2543,2016-03-30,2026-03-30,ok


Ortak başlangıç: 2023-09-21 00:00:00
Kullanılan varlık sayısı: 74
Toplam gün: 656 lookback: 252 test_months: 6


,AEFES,AGHOL,AKBNK,AKFYE,AKSA,AKSEN,ALARK,ARCLK,ASELS,ASTOR,BIMAS,BOBET,CCOLA,CIMSA,CWENE,DOAS,DOHOL,ECILC,EGEEN,EKGYO,ENERY,ENJSA,ENKAI,EREGL,EUPWR,FROTO,GARAN,GESAN,GUBRF,HALKB,HEKTS,ISCTR,ISMEN,KAYSE,KCAER,KCHOL,KLSER,KRDMD,MAVI,MGROS,MIATK,ODAS,OTKAR,OYAKC,PETKM,PGSUS,REEDR,SASA,SAHOL,SDTTR,SISE,SKBNK,SMRTG,SOKM,TAVHL,TCELL,THYAO,TKFEN,TOASO,TSKB,TTKOM,TTRAK,TUPRS,ULKER,VAKBN,VESBE,VESTL,YKBNK,ZOREN,USDTRY,EURTRY,ALTIN_TRY,GUMUS_TRY,PLATIN_TRY
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-03-24,17.320000,27.719999,68.116837,21.580000,10.97,74.800003,89.099998,114.000000,352.00,191.800003,679.5,19.170000,71.750000,48.959999,28.860001,193.500000,20.799999,115.199997,5692.5,19.700001,8.79,114.500000,93.699997,27.780001,35.000000,104.199997,130.000000,43.700001,476.50,38.060001,2.91,13.29,44.779999,4.44,11.40,182.669998,25.900000,29.280001,42.459999,586.0,38.540001,6.09,383.50,24.000000,19.549999,179.000000,7.93,2.38,89.949997,219.000000,47.439999,10.19,7.36,50.599998,291.50,106.199997,291.0,88.250000,276.25,11.524907,57.299999,453.00,259.000000,110.800003,32.279999,7.10,29.059999,33.340000,2.86,44.338600,51.498371,195058.795018,3071.512279,83888.631500
2026-03-25,17.639999,27.660000,68.698196,21.459999,10.81,76.500000,90.000000,114.500000,337.50,209.199997,705.0,19.280001,69.000000,50.450001,29.240000,195.600006,21.139999,115.900002,5715.0,20.400000,8.67,118.199997,93.949997,27.920000,37.680000,104.599998,130.399994,45.099998,471.75,37.820000,2.86,13.42,44.360001,4.36,11.27,186.699997,25.860001,29.440001,42.560001,604.5,37.700001,6.10,379.00,23.900000,19.030001,180.000000,7.69,2.36,91.150002,214.300003,47.160000,10.32,7.30,51.000000,305.00,107.400002,294.5,88.750000,278.50,11.496286,58.799999,451.25,250.500000,111.300003,32.520000,7.06,28.719999,34.040001,2.84,44.339298,51.500900,201734.930510,3208.435963,85362.014823
2026-03-26,17.190001,27.420000,67.250000,19.990000,11.25,74.699997,87.849998,111.300003,338.00,203.199997,673.0,19.059999,70.699997,49.840000,28.520000,193.199997,20.660000,112.000000,5660.0,19.809999,8.55,115.199997,91.849998,26.900000,37.720001,102.699997,126.699997,44.520000,473.50,36.660000,2.84,13.18,42.880001,4.30,10.99,189.300003,25.500000,28.920000,42.680000,600.5,36.919998,5.80,368.00,23.719999,19.170000,177.600006,7.46,2.39,89.949997,214.699997,45.320000,10.14,7.08,48.000000,294.50,106.000000,293.5,93.349998,272.50,11.124207,57.950001,453.50,240.199997,110.599998,31.299999,7.05,28.000000,33.180000,2.78,44.354599,51.379120,194073.547920,3001.519963,81537.061506
2026-03-27,16.730000,27.200001,66.900002,19.420000,10.79,71.599998,86.949997,110.500000,330.75,205.899994,670.5,19.030001,68.349998,49.099998,29.600000,192.899994,21.180000,109.000000,5620.0,19.270000,8.47,117.800003,92.449997,27.900000,37.599998,103.800003,126.500000,44.040001,462.75,36.799999,2.80,13.18,43.200001,4.26,11.00,192.600006,25.299999,29.400000,42.139999,603.0,37.599998,5.68,364.25,23.620001,19.200001,177.000000,7.14,2.33,89.099998,207.500000,44.880001,10.13,7.01,48.180000,292.75,106.400002,294.0,97.699997,279.75,11.100000,58.500000,450.00,240.500000,111.500000,31.040001,6.99,28.440001,33.180000,2.74,44.455551,51.317211,199694.335754,3091.661223,83158.552891
2026-03-30,16.280001,27.059999,65.900002,21.360001,10.71,73.949997,87.199997,109.900002,323.75,200.800003,678.5,18.950001,66.699997,48.740002,29.559999,190.300003,20.680000,108.800003,5537.5,19.340000,8.58,117.099998,93.300003,27.719999,40.759998,101.699997,125.900002,45.340000,469.00,36.099998,2.82,13.04,42.939999,4.29,10.88,194.600006,25.200001,29.120001,41.259998,596.0,39.860001,5.76,368.75,23.500000,20.440001,174.300003,7.09,2.32,89.050003,207.199997,43.520000,10.46,6.96,49.200001,291.50,105.000000,289.5,97.750000,273.00,11.080000,58.450001,446.25,246.300003,111.599998,30.299999,6.96,28.840000,32.880001,2.79,44.464699,51.013691,202990.251607,3137.873780,84616.321800


### Coverage charts nasıl okunur?

İlk grafik:
- her grupta kaç seri indirilebildiğini gösterir

İkinci grafik:
- indirilenler içinden kaçının gerçekten teste kaldığını gösterir

İdeal durumda:
- `ok` yüksek olmalı
- testte kalan sayısı da anlamlı büyüklükte olmalı

In [6]:
coverage_counts = coverage.groupby(['group', 'status']).size().reset_index(name='count')
fig = px.bar(coverage_counts, x='group', y='count', color='status', barmode='group', title='Veri kapsamı: grup bazında indirilen seriler')
fig.show()

eligible_counts = eligible_assets.groupby('group').size().reset_index(name='eligible_count')
fig = px.bar(eligible_counts, x='group', y='eligible_count', title='Teste kalan varlık sayısı')
fig.show()

## 4) Benchmark market charts

Burada strateji grafiği değil, önce piyasanın kendisi gösteriliyor.

Neden?
Çünkü portföy sonucunu yorumlamadan önce, temel rejimin ne olduğunu bilmek gerekir.

Göreceğin şeyler:
- BIST hisseleri genel olarak mı güçlüydü
- döviz mi baskındı
- altın / gümüş / platin ne yaptı
- bunlar birbirine ne kadar benziyordu

In [7]:
bist_cols = [c for c in prices.columns if c not in ['USDTRY', 'EURTRY', 'ALTIN_TRY', 'GUMUS_TRY', 'PLATIN_TRY']]
bist_basket = (1 + returns[bist_cols].mean(axis=1)).cumprod() if len(bist_cols) else pd.Series(dtype=float)
if len(bist_basket):
    bist_basket.name = 'BIST_BASKET'
bench = pd.concat([bist_basket, prices[['USDTRY', 'EURTRY', 'ALTIN_TRY', 'GUMUS_TRY', 'PLATIN_TRY']]], axis=1).dropna()
bench_norm = normalize_prices(bench.tail(504))
fig = px.line(bench_norm, x=bench_norm.index, y=bench_norm.columns, title='Son ~2 yıl benchmark görünümü (normalize)')
fig.show()

bench_rets = bench.pct_change().dropna()
fig = px.imshow(bench_rets.corr(), text_auto=True, aspect='auto', zmin=-1, zmax=1, color_continuous_scale='RdBu_r', title='Benchmark korelasyon ısı haritası')
fig.show()

## 5) Strategy definitions

Burada tekrar görünür hale gelen stratejiler:

### Equal
Bütün varlıklara eşit ağırlık verir.  
En sade baseline budur.

### HRP
Hierarchical Risk Parity.  
Amaç riski daha dengeli dağıtmaktır. Pratikte bazen daha sakin görünen varlıklara yönelebilir.

### BL
Black-Litterman.  
Burada kural tabanlı view üretip bunları optimizasyona sokuyoruz.

### HYBRID
HRP ve BL arasında bir blend.  
Amaç sadece tek yönteme bağımlı kalmamak.

Aşağıdaki bütün strateji grafikleri artık bu dört yaklaşımı birlikte karşılaştırır.

## 6) Monthly walk-forward backtest

Bu bölüm gerçek hayata daha yakın olan kısımdır.

Akış:
- her ay maaş gibi sabit katkı eklenir
- sadece o tarihe kadar olan veri kullanılır
- strateji yeni portföy kurar
- bir sonraki rebalance tarihine kadar tutulur

Bu yüzden bu notebook statik tek portföy değil, aylık yeniden kurulan portföyleri gösterir.

In [8]:
def get_rebalance_dates(prices, months):
    monthly = prices.resample('MS').first().dropna()
    month_starts = monthly.index[-months:]
    dates = []
    for dt in month_starts:
        idx = prices.index.searchsorted(dt)
        if idx < len(prices.index):
            dates.append(prices.index[idx])
    return pd.Index(pd.unique(pd.Index(dates)))

def run_strategy(strategy_name, prices, lookback, months):
    rebalance_dates = get_rebalance_dates(prices, months)
    cash = 0.0
    shares = pd.Series(0.0, index=prices.columns)
    hist, rebs, diags = [], [], []
    for i, dt in enumerate(rebalance_dates):
        cash += CONFIG['salary_try']
        window = prices.loc[:dt].tail(lookback)
        weights, diag = compute_weights(strategy_name, window)
        current = prices.loc[dt]
        total = float((shares * current).sum()) + cash
        shares = (weights * total) / current
        cash = 0.0
        nxt = rebalance_dates[i + 1] if i + 1 < len(rebalance_dates) else prices.index[-1]
        path = prices.loc[(prices.index >= dt) & (prices.index <= nxt)]
        vals = path.mul(shares, axis=1).sum(axis=1)
        for t, v in vals.items():
            hist.append({'date': t, 'strategy': strategy_name, 'portfolio_value': float(v)})
        row = {'date': dt, 'strategy': strategy_name, 'capital_after_contribution': total}
        for c, x in weights.items():
            row[f'weight_{c}'] = float(x)
        rebs.append(row)
        if diag is not None:
            tmp = diag.copy(); tmp['date'] = dt; tmp['strategy'] = strategy_name; diags.append(tmp)
    hist_df = pd.DataFrame(hist).drop_duplicates(['date', 'strategy'])
    weights_df = pd.DataFrame(rebs)
    diag_df = pd.concat(diags, ignore_index=True) if diags else pd.DataFrame()
    return hist_df, weights_df, diag_df

hist_parts, weight_parts, diag_parts = [], [], []
for s in CONFIG['strategies']:
    h, w, d = run_strategy(s, prices, lookback, months)
    hist_parts.append(h); weight_parts.append(w)
    if not d.empty:
        diag_parts.append(d)

hist_df = pd.concat(hist_parts, ignore_index=True)
weights_df = pd.concat(weight_parts, ignore_index=True)
diag_df = pd.concat(diag_parts, ignore_index=True) if diag_parts else pd.DataFrame()

display(hist_df.tail())
display(weights_df.tail())

,date,strategy,portfolio_value
507,2026-03-24,HYBRID,164128.005176
508,2026-03-25,HYBRID,164711.415264
509,2026-03-26,HYBRID,162301.012208
510,2026-03-27,HYBRID,162719.844868
511,2026-03-30,HYBRID,162915.100947


,date,strategy,capital_after_contribution,weight_AEFES,weight_AGHOL,weight_AKBNK,weight_AKFYE,weight_AKSA,weight_AKSEN,weight_ALARK,weight_ARCLK,weight_ASELS,weight_ASTOR,weight_BIMAS,weight_BOBET,weight_CCOLA,weight_CIMSA,weight_CWENE,weight_DOAS,weight_DOHOL,weight_ECILC,weight_EGEEN,weight_EKGYO,weight_ENERY,weight_ENJSA,weight_ENKAI,weight_EREGL,weight_EUPWR,weight_FROTO,weight_GARAN,weight_GESAN,weight_GUBRF,weight_HALKB,weight_HEKTS,weight_ISCTR,weight_ISMEN,weight_KAYSE,weight_KCAER,weight_KCHOL,weight_KLSER,weight_KRDMD,weight_MAVI,weight_MGROS,weight_MIATK,weight_ODAS,weight_OTKAR,weight_OYAKC,weight_PETKM,weight_PGSUS,weight_REEDR,weight_SASA,weight_SAHOL,weight_SDTTR,weight_SISE,weight_SKBNK,weight_SMRTG,weight_SOKM,weight_TAVHL,weight_TCELL,weight_THYAO,weight_TKFEN,weight_TOASO,weight_TSKB,weight_TTKOM,weight_TTRAK,weight_TUPRS,weight_ULKER,weight_VAKBN,weight_VESBE,weight_VESTL,weight_YKBNK,weight_ZOREN,weight_USDTRY,weight_EURTRY,weight_ALTIN_TRY,weight_GUMUS_TRY,weight_PLATIN_TRY
19,2025-11-03,HYBRID,50242.471720,0.001665,0.002123,0.013293,0.003223,0.009171,0.008000,0.002784,0.003463,0.034615,0.002761,0.006299,0.003151,0.002568,0.015546,0.002405,0.004121,0.004289,0.005878,0.003964,0.013096,0.023469,0.005591,0.003636,0.004004,0.002550,0.001931,0.000969,0.003213,0.003509,0.002581,0.002328,0.000965,0.001569,0.005177,0.003133,0.002134,0.002899,0.003025,0.006599,0.040072,0.002589,0.002062,0.003255,0.011611,0.003061,0.003080,0.002172,0.001941,0.001925,0.003853,0.004688,0.003489,0.002971,0.006949,0.023600,0.047150,0.003461,0.009754,0.002165,0.001779,0.002350,0.006736,0.012854,0.001637,0.003608,0.004163,0.003316,0.002586,0.002450,0.176136,0.176136,0.079465,0.067081,0.058158
20,2025-12-01,HYBRID,76891.275303,0.001775,0.002200,0.021733,0.003627,0.003993,0.003604,0.001897,0.002912,0.035663,0.002432,0.001747,0.003400,0.003912,0.003218,0.002532,0.003233,0.028337,0.003445,0.003770,0.002255,0.003592,0.007868,0.002727,0.002706,0.002075,0.018140,0.004712,0.002219,0.003198,0.002058,0.002494,0.004320,0.001852,0.004838,0.002735,0.001785,0.005234,0.001292,0.004308,0.039295,0.002313,0.002237,0.002151,0.010373,0.002151,0.002711,0.002954,0.002013,0.001732,0.003646,0.004107,0.003590,0.003048,0.003084,0.020472,0.041700,0.003277,0.004688,0.002093,0.004561,0.002888,0.006359,0.029214,0.003339,0.006833,0.002571,0.002041,0.002034,0.002585,0.180239,0.180239,0.080744,0.066339,0.058544
21,2026-01-02,HYBRID,106330.691183,0.002769,0.003450,0.009468,0.002660,0.004255,0.003290,0.002254,0.002695,0.022307,0.002465,0.002089,0.003172,0.026896,0.003108,0.002245,0.003712,0.041168,0.002843,0.003058,0.001905,0.006956,0.008474,0.004151,0.002373,0.001929,0.023445,0.004711,0.002281,0.003321,0.001906,0.002541,0.004155,0.001776,0.005107,0.003037,0.001598,0.004948,0.001745,0.024262,0.021878,0.002247,0.002187,0.002123,0.003527,0.002132,0.002295,0.002979,0.001270,0.001619,0.003497,0.001671,0.003197,0.002823,0.002229,0.023315,0.035072,0.003016,0.003139,0.002267,0.004303,0.002834,0.006116,0.038772,0.003335,0.011242,0.002603,0.002194,0.001961,0.002435,0.187415,0.187415,0.076562,0.056150,0.039654
22,2026-02-02,HYBRID,141281.297220,0.001877,0.015583,0.011515,0.002316,0.002806,0.021016,0.001885,0.002439,0.022820,0.001706,0.001160,0.004357,0.004305,0.002057,0.003234,0.002710,0.037999,0.004935,0.003733,0.000864,0.020990,0.019634,0.007809,0.001447,0.001425,0.002732,0.000829,0.002012,0.003626,0.001345,0.001823,0.001644,0.001680,0.008257,0.001598,0.001936,0.003583,0.001076,0.020111,0.032663,0.002922,0.001966,0.002566,0.002472,0.002946,0.002753,0.005606,0.001273,0.001637,0.008872,0.002165,0.003496,0.001880,0.002149,0.033151,0.041630,0.002817,0.017046,0.001890,0.000814,0.001841,0.004604,0.035315,0.001353,0.001840,0.002092,0.001749,0.002391,0.002703,0.197564,0.197564,0.071805,0.027508,0.026082
23,2026-03-02,HYBRID,167567.331542,0.001567,0.001983,0.010612,0.003624,0.001899,0.011535,0.002660,0.002492,0.023181,0.001660,0.030026,0.004039,0.010425,0.002080,0.003276,0.002877,0.050097

## 7) Summary table

Bu tablo en üst seviye sonucu verir.

Nasıl okunur:
- **Contributed**: toplam yatırılan para
- **Ending Value**: test sonundaki portföy değeri
- **Net Profit**: kalan kazanç
- **TWR**: burada pratik karşılaştırma metriği gibi okunmalı
- **Ann Vol**: dalgalanma seviyesi
- **Sharpe**: getiri / risk verimliliği gibi okunur
- **MaxDD**: en kötü düşüş

Tek başına bir sütuna bakma.
Getiri, drawdown ve ağırlık davranışını birlikte okumak gerekir.

In [9]:
rows = []
contributed = CONFIG['salary_try'] * months
for s in CONFIG['strategies']:
    sub = hist_df[hist_df['strategy'] == s].sort_values('date').copy()
    sub['ret'] = sub['portfolio_value'].pct_change().fillna(0.0)
    dd = sub['portfolio_value'] / sub['portfolio_value'].cummax() - 1
    rows.append({
        'Strategy': s,
        'Contributed': try_fmt(contributed),
        'Ending Value': try_fmt(sub['portfolio_value'].iloc[-1]),
        'Net Profit': try_fmt(sub['portfolio_value'].iloc[-1] - contributed),
        'TWR': pct_fmt(sub['portfolio_value'].iloc[-1] / contributed - 1),
        'Ann Vol': pct_fmt(sub['ret'].std() * np.sqrt(252)),
        'Sharpe': f"{(sub['ret'].mean() / (sub['ret'].std() + 1e-9) * np.sqrt(252)):.2f}",
        'MaxDD': pct_fmt(dd.min()),
    })
summary = pd.DataFrame(rows)
display(summary)

,Strategy,Contributed,Ending Value,Net Profit,TWR,Ann Vol,Sharpe,MaxDD
0,Equal,₺150.000,₺154.845,₺4.845,3.23%,165.97%,2.74,-9.12%
1,HRP,₺150.000,₺156.938,₺6.938,4.63%,164.87%,2.76,-1.08%
2,BL,₺150.000,₺169.018,₺19.018,12.68%,162.02%,2.89,-7.83%
3,HYBRID,₺150.000,₺162.915,₺12.915,8.61%,162.97%,2.83,-5.33%


### Performance comparison charts nasıl okunur?

İlk çizgi grafik:
- stratejilerin portföy değerini yan yana gösterir

İkinci grafik:
- aynı stratejilerin zirveden ne kadar düştüğünü gösterir

Üçüncü grafik:
- ay bazında hangi stratejinin daha iyi / kötü davrandığını görmeyi kolaylaştırır

In [10]:
fig = px.line(hist_df, x='date', y='portfolio_value', color='strategy', title='Strateji bazında portföy değeri')
fig.update_yaxes(tickprefix='₺', separatethousands=True)
fig.show()

dd_parts = []
for s in CONFIG['strategies']:
    sub = hist_df[hist_df['strategy'] == s].sort_values('date').copy()
    sub['drawdown'] = sub['portfolio_value'] / sub['portfolio_value'].cummax() - 1
    dd_parts.append(sub[['date', 'strategy', 'drawdown']])
dd_df = pd.concat(dd_parts, ignore_index=True)
fig = px.line(dd_df, x='date', y='drawdown', color='strategy', title='Strateji bazında drawdown')
fig.show()

mr_parts = []
for s in CONFIG['strategies']:
    sub = hist_df[hist_df['strategy'] == s].sort_values('date').copy().set_index('date')
    monthly = sub['portfolio_value'].resample('M').last().pct_change().dropna()
    mr_parts.append(pd.DataFrame({'date': monthly.index, 'strategy': s, 'monthly_return': monthly.values}))
mr_df = pd.concat(mr_parts, ignore_index=True)
fig = px.bar(mr_df, x='date', y='monthly_return', color='strategy', barmode='group', title='Aylık portföy getirileri')
fig.show()

## 8) Selection behavior charts

Bu bölüm stratejilerin gerçekte kaç varlık tuttuğunu ve hangi isimleri öne çıkardığını gösterir.

Neden önemli?
Çünkü bazen iyi görünen performans aslında çok dar ve riskli bir seçimin sonucudur.

In [11]:
active_counts = []
for s in CONFIG['strategies']:
    sub = weights_df[weights_df['strategy'] == s].copy()
    weight_cols = [c for c in sub.columns if c.startswith('weight_')]
    sub['active_asset_count'] = (sub[weight_cols] > 0).sum(axis=1)
    active_counts.append(sub[['date', 'strategy', 'active_asset_count']])
active_counts_df = pd.concat(active_counts, ignore_index=True)

fig = px.bar(active_counts_df, x='date', y='active_asset_count', color='strategy', barmode='group', title='Her rebalance anında aktif varlık sayısı')
fig.show()

last_weights = weights_df.sort_values('date').groupby('strategy').tail(1).copy()
latest_long = last_weights.melt(id_vars=['date', 'strategy', 'capital_after_contribution'], var_name='asset', value_name='weight')
latest_long = latest_long[latest_long['asset'].str.startswith('weight_')].copy()
latest_long['asset'] = latest_long['asset'].str.replace('weight_', '', regex=False)
latest_long = latest_long[latest_long['weight'] > 0].copy()

fig = px.bar(latest_long.sort_values(['strategy', 'weight'], ascending=[True, False]),
             x='asset', y='weight', color='strategy', facet_row='strategy',
             title='Son rebalance anında strateji bazında ağırlıklar')
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.show()

### Weight heatmap nasıl okunur?

Bu ısı haritası:
- zaman içinde hangi varlıkların öne çıktığını
- ağırlıkların sabit mi yoksa sık değişen mi olduğunu
- portföyün belirli birkaç isme yığılıp yığılmadığını

Okuma ipucu:
- koyu / yüksek hücreler sürekli aynı satırlarda toplanıyorsa, strateji daralıyor olabilir.

In [12]:
weight_cols = [c for c in weights_df.columns if c.startswith('weight_')]
cumw = weights_df[weight_cols].sum().sort_values(ascending=False).head(15).index.tolist()

heat = weights_df[['date', 'strategy'] + cumw].copy()
heat['date'] = heat['date'].dt.strftime('%Y-%m-%d')
heat['row'] = heat['strategy'] + ' | ' + heat['date']
heat = heat.set_index('row')[cumw]
heat.columns = [c.replace('weight_', '') for c in heat.columns]

fig = px.imshow(heat.T, aspect='auto', title='Ağırlık evrimi ısı haritası (en çok kullanılan 15 ağırlık sütunu)')
fig.update_xaxes(title='Strateji | Rebalance tarihi')
fig.update_yaxes(title='Varlık')
fig.show()

## 9) BL / HYBRID views

Bu tablo özellikle BL ve HYBRID içinde hangi view'ların üretildiğini gösterir.

Nasıl okunur:
- sürekli tavana vuran pozitif view'lar varsa, view sistemi fazla kaba olabilir
- sürekli aynı varlıklar pozitif kalıyorsa, model çok tek temalı olabilir

In [13]:
if len(diag_df):
    diag_show = diag_df.copy()
    diag_show['view_pct'] = diag_show['view'].map(pct_fmt)
    display(diag_show.tail(30))

,asset,view,view_source,date,strategy,view_pct
858,PETKM,0.105036,rules,2026-03-02,HYBRID,10.50%
859,PGSUS,-0.150000,rules,2026-03-02,HYBRID,-15.00%
860,REEDR,-0.150000,rules,2026-03-02,HYBRID,-15.00%
861,SASA,-0.150000,rules,2026-03-02,HYBRID,-15.00%
862,SAHOL,-0.050034,rules,2026-03-02,HYBRID,-5.00%
863,SDTTR,0.150000,rules,2026-03-02,HYBRID,15.00%
864,SISE,0.150000,rules,2026-03-02,HYBRID,15.00%
865,SKBNK,0.150000,rules,2026-03-02,HYBRID,15.00%
866,SMRTG,-0.150000,rules,2026-03-02,HYBRID,-15.00%
867,SOKM,0.150000,rules,2026-03-02,HYBRID,15.00%


## 10) Raw tables

Grafiklerden sonra detay tabloya bakmak istersen burada duruyor.

In [14]:
out = last_weights[['strategy'] + [c for c in last_weights.columns if c.startswith('weight_')]].copy()
out.columns = ['Strategy'] + [c.replace('weight_', '') for c in out.columns[1:]]
for c in out.columns[1:]:
    out[c] = out[c].map(pct_fmt)
display(out.reset_index(drop=True))

print('Weights by rebalance:')
display(weights_df)

print('Portfolio value path:')
display(hist_df.tail(20))

,Strategy,AEFES,AGHOL,AKBNK,AKFYE,AKSA,AKSEN,ALARK,ARCLK,ASELS,ASTOR,BIMAS,BOBET,CCOLA,CIMSA,CWENE,DOAS,DOHOL,ECILC,EGEEN,EKGYO,ENERY,ENJSA,ENKAI,EREGL,EUPWR,FROTO,GARAN,GESAN,GUBRF,HALKB,HEKTS,ISCTR,ISMEN,KAYSE,KCAER,KCHOL,KLSER,KRDMD,MAVI,MGROS,MIATK,ODAS,OTKAR,OYAKC,PETKM,PGSUS,REEDR,SASA,SAHOL,SDTTR,SISE,SKBNK,SMRTG,SOKM,TAVHL,TCELL,THYAO,TKFEN,TOASO,TSKB,TTKOM,TTRAK,TUPRS,ULKER,VAKBN,VESBE,VESTL,YKBNK,ZOREN,USDTRY,EURTRY,ALTIN_TRY,GUMUS_TRY,PLATIN_TRY
0,HRP,0.10%,0.13%,0.10%,0.24%,0.13%,0.28%,0.18%,0.17%,0.10%,0.11%,0.14%,0.27%,0.15%,0.14%,0.22%,0.19%,0.22%,0.30%,0.26%,0.06%,0.43%,0.19%,0.16%,0.15%,0.07%,0.17%,0.05%,0.10%,0.27%,0.06%,0.12%,0.10%,0.11%,0.57%,0.10%,0.12%,0.26%,0.12%,0.22%,0.12%,0.22%,0.13%,0.13%,0.11%,0.19%,0.15%,0.25%,0.09%,0.11%,0.07%,0.11%,0.22%,0.09%,0.15%,0.11%,0.15%,0.18%,0.44%,0.12%,0.05%,0.13%,0.28%,0.23%,0.13%,0.08%,0.14%,0.11%,0.15%,0.09%,70.61%,15.78%,1.40%,0.27%,0.51%
1,Equal,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%,1.35%
2,BL,0.00%,0.00%,1.84%,0.00%,0.00%,1.47%,0.00%,0.00%,4.35%,0.00%,5.61%,0.00%,1.64%,0.00%,0.00%,0.00%,9.39%,0.00%,0.00%,0.00%,3.00%,2.22%,4.14%,2.84%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,1.64%,0.00%,0.00%,0.00%,0.00%,0.00%,7.13%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,4.96%,0.00%,1.40%,0.00%,0.35%,0.88%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,10.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,10.00%,10.00%,10.00%,3.80%,3.37%
3,HYBRID,0.16%,0.20%,1.06%,0.36%,0.19%,1.15%,0.27%,0.25%,2.32%,0.17%,3.00%,0.40%,1.04%,0.21%,0.33%,0.29%,5.01%,0.44%,0.39%,0.09%,2.14%,1.39%,2.30%,1.65%,0.10%,0.26%,0.08%,0.16%,0.40%,0.09%,0.18%,0.15%,0.17%,1.67%,0.16%,0.18%,0.39%,0.18%,0.33%,3.72%,0.32%,0.20%,0.20%,0.16%,0.28%,0.22%,0.38%,0.14%,0.16%,2.58%,0.17%,1.02%,0.14%,0.40%,0.61%,0.23%,0.26%,0.66%,0.18%,0.08%,0.19%,0.42%,5.33%,0.19%,0.12%,0.20%,0.17%,0.23%,0.13%,19.92%,19.92%,7.07%,2.30%,2.44%


Weights by rebalance:


,date,strategy,capital_after_contribution,weight_AEFES,weight_AGHOL,weight_AKBNK,weight_AKFYE,weight_AKSA,weight_AKSEN,weight_ALARK,weight_ARCLK,weight_ASELS,weight_ASTOR,weight_BIMAS,weight_BOBET,weight_CCOLA,weight_CIMSA,weight_CWENE,weight_DOAS,weight_DOHOL,weight_ECILC,weight_EGEEN,weight_EKGYO,weight_ENERY,weight_ENJSA,weight_ENKAI,weight_EREGL,weight_EUPWR,weight_FROTO,weight_GARAN,weight_GESAN,weight_GUBRF,weight_HALKB,weight_HEKTS,weight_ISCTR,weight_ISMEN,weight_KAYSE,weight_KCAER,weight_KCHOL,weight_KLSER,weight_KRDMD,weight_MAVI,weight_MGROS,weight_MIATK,weight_ODAS,weight_OTKAR,weight_OYAKC,weight_PETKM,weight_PGSUS,weight_REEDR,weight_SASA,weight_SAHOL,weight_SDTTR,weight_SISE,weight_SKBNK,weight_SMRTG,weight_SOKM,weight_TAVHL,weight_TCELL,weight_THYAO,weight_TKFEN,weight_TOASO,weight_TSKB,weight_TTKOM,weight_TTRAK,weight_TUPRS,weight_ULKER,weight_VAKBN,weight_VESBE,weight_VESTL,weight_YKBNK,weight_ZOREN,weight_USDTRY,weight_EURTRY,weight_ALTIN_TRY,weight_GUMUS_TRY,weight_PLATIN_TRY
0,2025-10-01,Equal,25000.000000,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514
1,2025-11-03,Equal,50026.762332,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514
2,2025-12-01,Equal,74842.808006,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514
3,2026-01-02,Equal,102230.786519,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514
4,2026-02-02,Equal,139814.135603,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0.013514,0

Portfolio value path:


,date,strategy,portfolio_value
492,2026-03-03,HYBRID,164851.426977
493,2026-03-04,HYBRID,164499.050122
494,2026-03-05,HYBRID,165590.516342
495,2026-03-06,HYBRID,164744.189775
496,2026-03-09,HYBRID,163790.032110
497,2026-03-10,HYBRID,167511.747939
498,2026-03-11,HYBRID,167068.975538
499,2026-03-12,HYBRID,167767.127068
500,2026-03-13,HYBRID,166667.702468
501,2026-03-16,HYBRID,165573.545591
